In [ ]:
from huggingface_hub import model_info, hf_hub_download

from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
from nltk.corpus import brown
from datasets import Dataset

model_name = "nevertipyourlandlord/distilbert-brown-pos"

# If this runs, I fine-tuned DistilBERT (can check on web too)
print(model_info(model_name))

ModelInfo(id='nevertipyourlandlord/distilbert-brown-pos', author='nevertipyourlandlord', base_models=None, card_data=None, children_model_count=None, config={'architectures': ['DistilBertForTokenClassification'], 'model_type': 'distilbert', 'tokenizer_config': {'cls_token': '[CLS]', 'mask_token': '[MASK]', 'pad_token': '[PAD]', 'sep_token': '[SEP]', 'unk_token': '[UNK]'}}, created_at=datetime.datetime(2026, 4, 26, 22, 42, 1, tzinfo=datetime.timezone.utc), disabled=False, downloads=0, downloads_all_time=None, eval_results=None, gated=False, gguf=None, inference=None, inference_provider_mapping=None, last_modified=datetime.datetime(2026, 4, 26, 22, 43, 41, tzinfo=datetime.timezone.utc), library_name=None, likes=1, mask_token='[MASK]', model_index=None, pipeline_tag=None, private=False, resource_group=None, safetensors=SafeTensorsInfo(parameters={'F32': 66443625}, total=66443625), security_repo_status=None, sha='ab4cfa546cee300341baec6aafeda7a50544b610', siblings=[RepoSibling(rfilename='.

In [2]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [3]:
model = AutoModelForTokenClassification.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [4]:
model.eval()

DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
   

In [ ]:
#This cell sets up evaluation for the fine-tuned model, using definitions in the fine-tuning notebook.

def simplify_tag(tag):
    return tag.split("+")[0].split("-")[0]

tagged_sentences = list(brown.tagged_sents())

brown_tags = set(simplify_tag(tag) for sent in tagged_sentences for word, tag in sent)

tag_to_id = {tag: i for i, tag in enumerate(sorted(brown_tags))}
id_to_tag = {i: tag for tag, i in tag_to_id.items()}

data = [
    {
        "tokens": [w for w, _ in sent],
        "tags": [simplify_tag(t) for _, t in sent],
    }
    for sent in tagged_sentences
]

dataset = Dataset.from_list(data).train_test_split(test_size=0.2, seed=67)

test_dataset = dataset["test"]